In [203]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/vedauv/what-makes-products-sell-15k-e-commerce-listings/datasheet.md
/kaggle/input/datasets/vedauv/what-makes-products-sell-15k-e-commerce-listings/README.md
/kaggle/input/datasets/vedauv/what-makes-products-sell-15k-e-commerce-listings/product_sales.csv


In [204]:
train = pd.read_csv('/kaggle/input/datasets/vedauv/what-makes-products-sell-15k-e-commerce-listings/product_sales.csv')

In [205]:
train.shape

(15000, 13)

In [206]:
train.columns

Index(['product_id', 'category', 'brand', 'price_usd', 'discount_pct',
       'rating', 'num_reviews', 'seller_rating', 'delivery_days',
       'free_shipping', 'in_stock', 'units_sold', 'high_seller'],
      dtype='object')

In [207]:
train.isnull().sum()

product_id       0
category         0
brand            0
price_usd        0
discount_pct     0
rating           0
num_reviews      0
seller_rating    0
delivery_days    0
free_shipping    0
in_stock         0
units_sold       0
high_seller      0
dtype: int64

In [208]:
train.head()

,product_id,category,brand,price_usd,discount_pct,rating,num_reviews,seller_rating,delivery_days,free_shipping,in_stock,units_sold,high_seller
0,prod_000001,Jewelry,Brand_115,192.29,37.8,4.24,237,4.21,2,1,1,13,0
1,prod_000002,Home & Kitchen,Brand_045,68.98,7.1,4.60,37,4.07,2,1,0,18,0
2,prod_000003,Health,Brand_120,13.51,19.7,3.98,19,3.94,7,1,1,13,0
3,prod_000004,Automotive,Brand_333,69.35,20.9,4.24,4184,4.35,10,1,1,42,0
4,prod_000005,Pet Supplies,Brand_143,28.14,42.8,4.18,36,4.29,9,0,1,6,0


In [209]:
train['category'].value_counts()

category
Toys              1040
Groceries         1036
Jewelry           1030
Pet Supplies      1023
Office            1022
Garden            1019
Baby              1006
Electronics       1000
Clothing           992
Books              982
Health             980
Sports             979
Beauty             970
Home & Kitchen     966
Automotive         955
Name: count, dtype: int64

In [210]:
train = pd.get_dummies(train,columns=['category'],drop_first=True)

In [211]:
features = ['price_usd','discount_pct','rating','num_reviews','seller_rating','delivery_days','free_shipping','in_stock',
            'category_Baby',
           'category_Beauty', 'category_Books', 'category_Clothing',
           'category_Electronics', 'category_Garden', 'category_Groceries',
           'category_Health', 'category_Home & Kitchen', 'category_Jewelry',
           'category_Office', 'category_Pet Supplies', 'category_Sports',
           'category_Toys']

In [212]:
X = train[features]

In [213]:
y = train['units_sold']

In [214]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

In [215]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42
)

In [216]:
lin_reg = LinearRegression()
lin_reg.fit(X_train,y_train)


LinearRegression()

In [217]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [218]:
y_pred = np.round(lin_reg.predict(X_test),0)
print("R2: ",r2_score(y_test,y_pred))
print("MAE: ",mean_absolute_error(y_test,y_pred))
print("MSE: ",mean_squared_error(y_test,y_pred))


R2:  0.27346524705191644
MAE:  14.1904
MSE:  427.1008


In [220]:
y_train_log = np.log1p(y_train)
lin_reg.fit(X_train, y_train_log)
pred_log = lin_reg.predict(X_test)
y_pred_log = np.expm1(pred_log)

print("R2:", r2_score(y_test, y_pred_log))
print("MAE:", mean_absolute_error(y_test, y_pred_log))
print("MSE:", mean_squared_error(y_test, y_pred_log))

R2: -553541.6915614808
MAE: 309.2851582257026
MSE: 325405667.7134006


In [226]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("R2:", r2_score(y_test, y_pred_rf))
print("MAE:", mean_absolute_error(y_test, y_pred_rf))
print("MSE:", mean_squared_error(y_test, y_pred_rf))

R2: 0.4428195920796637
MAE: 12.4624
MSE: 327.5441360533333


In [231]:
from xgboost import XGBRegressor
xgb_reg = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    objective='reg:squarederror',
    random_state=42,
    n_jobs=-1
)

xgb_reg.fit(X_train, y_train)

y_pred_xgb = xgb_reg.predict(X_test)

print("R2:", r2_score(y_test, y_pred_xgb))
print("MAE:", mean_absolute_error(y_test, y_pred_xgb))
print("MSE:", mean_squared_error(y_test, y_pred_xgb))

R2: 0.45856738090515137
MAE: 12.2001314163208
MSE: 318.2866516113281
